# Entity class distributions

Per-split (train/validation/test) token-level entity class counts for the two dataset versions in `../data`, with B-/I- prefixes merged into their base class (e.g. `B-PESSOA` and `I-PESSOA` both count as `PESSOA`).

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from datasets import load_from_disk, load_dataset

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

## Load datasets

In [ ]:
DATASETS = {
    "lenerbr_sentence_deduped": load_from_disk("../data/lenerbr_sentence_deduped"),
    "lenerbr_entity_exemplar_deduped": load_from_disk("../data/lenerbr_entity_exemplar_deduped"),
    "lenerbr_random_downsample": load_from_disk("../data/lenerbr_random_downsample"),
    "lenerbr": load_dataset("peluz/lener_br"),
}

LABEL_NAMES = next(iter(DATASETS.values()))["train"].features["ner_tags"].feature.names
SPLITS = ["train", "validation", "test"]
BASE_ORDER = list(dict.fromkeys(
    tag if tag == "O" else tag.split("-", 1)[1] for tag in LABEL_NAMES
))

In [ ]:
def base_class(tag: str) -> str:
    """Strips the B-/I- prefix, e.g. 'B-PESSOA' -> 'PESSOA', 'O' -> 'O'."""
    return tag if tag == "O" else tag.split("-", 1)[1]


def per_split_base_table(ds) -> pd.DataFrame:
    """Base entity class counts (B-/I- merged) per split, plus a total column."""
    data = {}
    for split in SPLITS:
        counts = Counter()
        for tags in ds[split]["ner_tags"]:
            for tag_id in tags:
                counts[base_class(LABEL_NAMES[tag_id])] += 1
        data[split] = [counts.get(cls, 0) for cls in BASE_ORDER]
    df = pd.DataFrame(data, index=BASE_ORDER)
    df["total"] = df[SPLITS].sum(axis=1)
    df.index.name = "class"
    df.loc["TOTAL"] = df.sum()
    return df

In [ ]:
def add_percentages(df: pd.DataFrame) -> pd.DataFrame:
    """Adds a '<col> %' column after each count column, i.e. each class's share of that column's TOTAL."""
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        out[col] = df[col]
        out[f"{col} %"] = (df[col] / df.loc["TOTAL", col] * 100).round(2)
    return out

`lenerbr_sentence_deduped` per-split distribution (BI grouped)

In [ ]:
df_filtered = per_split_base_table(DATASETS["lenerbr_sentence_deduped"])
df_filtered

In [ ]:
print(add_percentages(df_filtered))

`lenerbr_entity_exemplar_deduped` per-split distribution (BI grouped)

In [ ]:
df_entity_filtered = per_split_base_table(DATASETS["lenerbr_entity_exemplar_deduped"])
df_entity_filtered

In [ ]:
print(add_percentages(df_entity_filtered))

`lenerbr_random_downsample` per-split distribution (BI grouped)

In [ ]:
df_random_sample = per_split_base_table(DATASETS["lenerbr_random_downsample"])
df_random_sample

In [ ]:
print(add_percentages(df_random_sample))

`lenerbr` per-split distribution (BI grouped)

In [ ]:
df = per_split_base_table(DATASETS["lenerbr"])
df

In [ ]:
print(add_percentages(df))

In [ ]:
sns.set_theme(style="white")

dfs_by_version = {
    "LeNER-Br": df,
    "w/o Sentence Leakage": df_filtered,
    "w/o Entity Exemplar Leakage": df_entity_filtered,
    "Random Sample": df_random_sample,
}

long_df = pd.concat([
    table.loc[BASE_ORDER, "total"]
    .rename("count")
    .rename_axis("Entity Class")
    .reset_index()
    .assign(dataset=name)
    for name, table in dfs_by_version.items()
])

plt.figure(figsize=(12, 4))
ax = sns.barplot(data=long_df, x="Entity Class", y="count", hue="dataset")
ax.set_yscale("log")
ax.set_ylabel("Count (log)")
# ax.set_title("Entity class distribution by dataset version")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Dataset version")
plt.tight_layout()
plt.savefig("../results/entity_distribution.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def kl_divergence(p: pd.Series, q: pd.Series) -> float:
    p, q = p.align(q)
    support = p > 0
    return float(np.sum(p[support] * np.log(p[support] / q[support])))


proportions_by_version = {
    name: table.loc[BASE_ORDER, "total"] / table.loc[BASE_ORDER, "total"].sum()
    for name, table in dfs_by_version.items()
}

versions = list(proportions_by_version)
kl_matrix = pd.DataFrame(
    [[kl_divergence(proportions_by_version[p], proportions_by_version[q]) for q in versions] for p in versions],
    index=versions,
    columns=versions,
)
kl_matrix.index.name = "P"
kl_matrix.columns.name = "Q"
kl_matrix